<a href="https://colab.research.google.com/github/nilum2002/Fine-Tune-LLMs-/blob/Main/gemma2-instruct-sinfintune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -U bitsandbytes==0.49.0
!pip install -q -U peft==0.18.0
!pip install -q -U trl==0.26.2
!pip install -q -U accelerate
!pip install -q -U datasets==4.4.2
!pip install -q -U transformers==4.57.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 129.8 MB/s eta 0:00:00


In [3]:
import os
import transformers
import torch
from google.colab import userdata
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM # for generating some text based on decoder based transformer
from transformers import BitsAndBytesConfig, GemmaTokenizer


In [4]:
from google.colab import userdata
import os


os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [5]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/Thimira/sinhala-llm-dataset-llama-prompt-format/" + splits["train"])

In [7]:
model_id = "google/gemma-2b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # all 32 bit weights converts in to 4 bits
    bnb_4bit_quant_type="nf4", # 4-bit normal Float
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token = os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config,
                                             token = os.environ["HF_TOKEN"],
                                             device_map={"":0}
)


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [9]:

lora_config = LoraConfig(
    r = 8,
    target_modules = ["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type = "CAUSAL_LM"
)

In [29]:
df[['prompt', 'response']] = df['text'].apply(lambda x: pd.Series(split_text(x)))
df = df.drop(columns=['text'])
print(df.head())

                                              prompt  \
0  කේතීකරණ ව්‍යාපෘතියක් සමඟ ඔබ විසඳූ ගැටලුවක් විස...   
1  පහත මාතෘකාව සම්බන්ධයෙන් විස්තර සපයන්න: සිසු ගැ...   
2  පහත මාතෘකාව සම්බන්ධයෙන් විස්තර සපයන්න: ශ්‍රී ල...   
3  පහත මාතෘකාව සම්බන්ධයෙන් විස්තර සපයන්න: පාර්ලිම...   
4  පහත මාතෘකාව සම්බන්ධයෙන් විස්තර සපයන්න: ස්ථාන ම...   

                                            response  
0  AI ලෙස, මම කේතීකරණ ව්‍යාපෘති සමඟ බොහෝ ගැටලු වි...  
1  ප්‍රහාරයෙන් තුවාල ලැබූ පාසල් සිසුවා රෝහලේ ප්‍ර...  
2  ශ්‍රී ලංකාව හා සිම්බාබ්වේ අතර පැවැත්වෙන ලෝක කු...  
3  පවතින දේශපාලන ක්‍රමවේදය තුළින් තවදුරටත් රට ගොඩ...  
4  ජාතික ගුරු මාරු ප්‍රතිපත්තිය උල්ලංඝණය කරමින් ස...  


In [30]:
from datasets import DatasetDict

dataset = Dataset.from_pandas(df)

splitted_dataset = dataset.train_test_split(test_size=0.1, seed=42)

data = DatasetDict({
    "train": splitted_dataset["train"],
    "test": splitted_dataset["test"]
})

print(data)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response'],
        num_rows: 188645
    })
    test: Dataset({
        features: ['prompt', 'response'],
        num_rows: 20961
    })
})


In [39]:
def formatting_func_str(example):
    return f"<s>[INST] {example['prompt']} [/INST] {example['response']}</s>"

# The dataset already has the 'text' column as per the error message, so this map operation is not needed.
# If this cell were to be run from a fresh state (where data["train"] had 'prompt' and 'response'),
# this line would be necessary. However, to fix the current ValueError from a rerun,
# we assume the transformation has already taken place.
# data["train"] = data["train"].map(lambda example: {"text": formatting_func_str(example)}, remove_columns=["prompt", "response"])

# Re-initialize the trainer with the pre-processed dataset (which now only has 'text')
trainer = SFTTrainer(
    model = model,
    train_dataset = data["train"],
    args = transformers.TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = False,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "paged_adamw_8bit"
    ),
    peft_config = lora_config,
    # The formatting_func is no longer needed in SFTTrainer's arguments if the dataset is already pre-formatted to 'text'
    # as SFTTrainer will look for the 'text' column by default.
    # However, if it were to do its own internal processing, having formatting_func might reintroduce previous errors.
    # The error message implied data["train"] is already only ['text'], so we can rely on that.
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/188645 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/188645 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/188645 [00:00<?, ? examples/s]

**Reasoning**:
The `SFTTrainer` has been successfully initialized in the previous step. The next logical action is to start the fine-tuning process by calling the `train()` method on the `trainer` object.



In [40]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
1,3.762500
2,3.444700
3,2.933200
4,3.372300
5,2.604800
6,2.781900
7,2.752200
8,2.512100
9,2.576000
10,2.072300


wandb: WARNING URL not available in offline run


TrainOutput(global_step=100, training_loss=1.7407583749294282, metrics={'train_runtime': 624.0581, 'train_samples_per_second': 0.641, 'train_steps_per_second': 0.16, 'total_flos': 1381842655272960.0, 'train_loss': 1.7407583749294282, 'epoch': 0.002120384849850248})

In [41]:
model_name = "gemma-2b-it-sinhala-finetune"
trainer.push_to_hub(f"Nilum/{model_name}", private=True)

TypeError: BaseTrainer.create_model_card() got an unexpected keyword argument 'private'

In [42]:
model_name = "gemma-2b-it-sinhala-finetune"
trainer.push_to_hub(f"Nilum/{model_name}")

wandb: WARNING URL not available in offline run


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt/outputs/tokenizer.json: 100%|##########| 34.4MB / 34.4MB            

  ...t/outputs/tokenizer.model: 100%|##########| 4.24MB / 4.24MB            

  ...adapter_model.safetensors:   0%|          | 27.0kB / 19.6MB            

  ...14782.352fa3dbb3a3.1404.0:   3%|3         | 1.37kB / 43.9kB            

  ...outputs/training_args.bin:   3%|3         |   195B / 6.22kB            

CommitInfo(commit_url='https://huggingface.co/Nilum/outputs/commit/d3ceb1bd21d622b6410b7eece1115cd040e616c6', commit_message='Nilum/gemma-2b-it-sinhala-finetune', commit_description='', oid='d3ceb1bd21d622b6410b7eece1115cd040e616c6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nilum/outputs', endpoint='https://huggingface.co', repo_type='model', repo_id='Nilum/outputs'), pr_revision=None, pr_num=None)

In [2]:
sinhala_question = "ශ්‍රී ලංකාවේ අගනුවර කුමක්ද?" # What is the capital of Sri Lanka?
test_prompt = f"<s>[INST] {sinhala_question} [/INST]"
print(test_prompt)

<s>[INST] ශ්‍රී ලංකාවේ අගනුවර කුමක්ද? [/INST]


In [3]:
input_ids = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**input_ids, max_new_tokens=100)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Original Prompt:")
print(test_prompt)
print("\nModel Response:")
print(response)

NameError: name 'tokenizer' is not defined